## Tracing LangChain with MLflow

### Installing Utilities and Libraries

In [1]:
%pip install langchain-anthropic==1.5.4 langchain==1.3.14 langgraph==1.2.10 mlflow==3.15.2

Looking in indexes: https://pypi.org/simple/

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
anthropic_api_key = os.getenv("CLAUDE_API_KEY")
anthropic_model_name = os.getenv("CLAUDE_MODEL_NAME")

### Enable MLflow Tracing

In [3]:
import mlflow

# Calling autolog for LangChain will enable trace logging.
mlflow.langchain.autolog()

# Optional: Set a tracking URI and an experiment
mlflow.set_experiment("CCAR-P Experiment v2")
mlflow.set_tracking_uri("http://localhost:5001")

/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/09/27 12:40:48 INFO mlflow.tracking.fluent: Experiment with name 'CCAR-P Experiment v2' does not exist. Creating a new experiment.


### Implement the Weather API Caller

In [4]:
import requests
from typing_extensions import Annotated

def get_current_weather(
        location: Annotated[str, ..., "Location for which the weather condition needs to be fetched"]
    ):

    response = requests.get(
        f"https://wttr.in/{location}",
        params={
            "format": "j1"
        }
    )

    response.raise_for_status()

    return response.json()

### Instantiate the ChatAnthropic Class

In [5]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(
    model_name = anthropic_model_name,
    api_key = anthropic_api_key,
)

### Bind the Model with Tools and Execute

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

model_with_tools = model.bind_tools([get_current_weather])

# User message
messages = [
    HumanMessage("What is the weather of London currently like?")
]

# First model call
response = model_with_tools.invoke(messages)

# Add assistant message containing tool call
messages.append(response)

# Execute every requested tool
for tool_call in response.tool_calls:

    if tool_call["name"] == "get_current_weather":

        result = get_current_weather(**tool_call["args"])

        messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )

# Second model call (LLM now sees tool output)
final_response = model_with_tools.invoke(messages)

print(final_response.content)

[HumanMessage(content='What is the weather of London currently like?', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'id': 'toolu_015vTyRzgoncnV65u1csebkP', 'caller': {'type': 'direct'}, 'input': {'location': 'London'}, 'name': 'get_current_weather', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CfTY1kQ2M6XHxGCr2pRdr', 'container': None, 'model': 'claude-sonnet-5', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 451, 'output_tokens': 50, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': None, 'service_tier': 'standard'}, 'diagnostics': None, 'model_name': 'claude-sonnet-5', 'model_provider': 'anthropic'}, id='lc_run--01a0e1b9-e726-71e2-ad70-23f6836839f5-0', tool_calls=[{'name': 'get_current_weather

Trace(trace_id=tr-4d6123d6d7e2447b7105e191f67b8f57)